# GPT-5.6 Agentic Capabilities — the Agents SDK

This notebook covers the **OpenAI Agents SDK**: the typed, server-side framework where *your* code owns orchestration, tool execution, state, and approvals. Recommended agent model: **`gpt-5.6-sol`** (bare `gpt-5.6` alias also routes to Sol).

We cover the primitives in dependency order: **Agent -> Runner -> Tools -> Handoffs -> Guardrails -> Tracing.**

> **Note on the older hosted Agent Builder:** the visual Agent Builder was deprecated (announced Jun 3 2026, shuts down Nov 30 2026). The migration path is **this SDK**. Don't teach Agent Builder.

> **Agents SDK — import surface verified.** Package: `openai-agents` → imports from `agents`. All imports in this notebook (`Agent`, `Runner`, `function_tool`, `input_guardrail`, `output_guardrail`, `GuardrailFunctionOutput`, `trace`) are confirmed against the current SDK. Install the latest version before running: `pip install -U openai-agents`.

## Install

In [1]:
# Install the Agents SDK
!pip install -U openai-agents

## 1. Agent — the LLM config (instructions + tools)

In [2]:
from agents import Agent

assistant = Agent(
    name="Assistant",
    model="gpt-5.6-sol",
    instructions="Help users with their tasks. Be concise and pin your reasoning to evidence.",
)

## 2. Runner — execute the agent loop (streaming)

In [ ]:
import asyncio
from agents import Runner

# Runner.run_streamed() returns a streaming run you can iterate event-by-event.
async def main():
    result = Runner.run_streamed(assistant, "Explain what an LLM agent is in 3 bullets.")
    async for event in result.stream_events():
        print(event)
    print("\nFINAL:", result.final_output)

# In a notebook: await main()  (or asyncio.run(main()) in a script)
await main()

## 3. Tools — function tools + hosted tools (`shell`, `web_search`)

Function tools wrap your Python callables; hosted tools (`shell`, `web_search`) run on OpenAI's side.

In [4]:
from agents import Agent, function_tool

@function_tool
def add(x: int, y: int) -> int:
    """Add two integers."""
    return x + y

research_agent = Agent(
    name="Researcher",
    model="gpt-5.6-sol",
    instructions="Research questions and compute with the add tool when needed.",
    tools=[add],
)

result = Runner.run_streamed(research_agent, "What's 21 + 21?")
async for event in result.stream_events():
    pass
print(result.final_output)

42


## 4. Handoffs — delegate to a specialist agent

A triage agent routes work to specialists; you control which agent ultimately replies.

In [5]:
from agents import Agent, Runner

refunds_agent = Agent(
    name="Refunds Specialist",
    model="gpt-5.6-sol",
    instructions="Handle refund requests. Confirm order id, then explain the refund timeline.",
)

billing_agent = Agent(
    name="Billing Specialist",
    model="gpt-5.6-sol",
    instructions="Answer billing and invoice questions precisely.",
)

triage_agent = Agent(
    name="Triage",
    model="gpt-5.6-sol",
    instructions="Route the user to the right specialist. Hand off; do not answer directly.",
    handoffs=[refunds_agent, billing_agent],
)

result = Runner.run_streamed(triage_agent, "I want a refund for order #A1043.")
async for _ in result.stream_events():
    pass
print(result.final_output)

Tool name 'transfer_to_Refunds Specialist' contains invalid characters for function calling and has been transformed to 'transfer_to_refunds_specialist'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Billing Specialist' contains invalid characters for function calling and has been transformed to 'transfer_to_billing_specialist'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Refunds Specialist' contains invalid characters for function calling and has been transformed to 'transfer_to_refunds_specialist'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Billing Specialist' contains invalid characters for function calling and has been transformed to 'transfer_to_billing_specialist'. Please use only letters, digits, and underscores to avoid potential naming conflicts.


I can help with that. I’ve noted the order ID: **#A1043**.

Refunds typically take **5–10 business days** to appear back on your original payment method after the refund is processed, depending on your bank or card issuer.


## 5. Input Guardrail — validate/block before the agent runs

Input guardrails inspect the user input and can block or pause risky requests (e.g. add a human-review checkpoint).

In [10]:
from agents import Agent, Runner, input_guardrail, GuardrailFunctionOutput

@input_guardrail
async def block_secrets(ctx, agent, input) -> GuardrailFunctionOutput:
    """Block prompts that look like they contain credentials."""
    text = input if isinstance(input, str) else str(input)
    tripwire = "api_key" in text.lower() or "password" in text.lower()
    return GuardrailFunctionOutput(
        output_info={"reason": "looks like a secret"} if tripwire else {},
        tripwire_triggered=tripwire,
    )

guarded_agent = Agent(
    name="Guarded Assistant",
    model="gpt-5.6-sol",
    instructions="Help with safe requests only.",
    input_guardrails=[block_secrets],
)

# This should trip the input guardrail:
try:
    result = Runner.run_streamed(guarded_agent, "Here is my password: hunter2, log me in.")
    async for _ in result.stream_events():
        pass
    print(result.final_output)
except Exception as e:
    print("Input guardrail blocked the run:", type(e).__name__, e)

Input guardrail blocked the run: InputGuardrailTripwireTriggered Guardrail InputGuardrail triggered tripwire


## 6. Output Guardrail — validate the agent's answer before returning

Output guardrails inspect the produced answer (e.g. ensure no PII / no overconfident claims) and can block it.

In [11]:
from agents import Agent, Runner, output_guardrail, GuardrailFunctionOutput

@output_guardrail
async def no_absolute_promises(ctx, agent, output) -> GuardrailFunctionOutput:
    """Block answers that make absolute guarantees."""
    banned = ("guaranteed", "100% safe", "never fails")
    hit = any(b in str(output).lower() for b in banned)
    return GuardrailFunctionOutput(
        output_info={"reason": "absolute promise"} if hit else {},
        tripwire_triggered=hit,
    )

careful_agent = Agent(
    name="Careful Assistant",
    model="gpt-5.6-sol",
    instructions="Answer helpfully but never make absolute guarantees.",
    output_guardrails=[no_absolute_promises],
)

try:
    result = Runner.run_streamed(careful_agent, "Is this investment safe?")
    async for _ in result.stream_events():
        pass
    print(result.final_output)
except Exception as e:
    print("Output guardrail blocked the answer:", type(e).__name__, e)

Output guardrail blocked the answer: OutputGuardrailTripwireTriggered Guardrail OutputGuardrail triggered tripwire


## 7. Tracing — view, debug, and optimize runs

The SDK records traces of every agent run. Wrap work in a `trace(...)` context to group it.

**Opening the trace viewer:** traces are sent to the OpenAI platform — open the **Traces** dashboard at **https://platform.openai.com/traces** to inspect each run (agent steps, tool calls, handoffs, guardrail trips). From there you can progress from traces to eval loops. *(Confirm the exact dashboard path in your account — the hosted Evals platform is being deprecated, so prefer the SDK tracing -> code-based eval workflow.)*

In [8]:
from agents import trace, Runner

with trace("course-agentic-demo"):
    result = Runner.run_streamed(assistant, "Summarize this notebook's agent primitives in 5 bullets.")
    async for _ in result.stream_events():
        pass
    print(result.final_output)

print("\nOpen https://platform.openai.com/traces to inspect the 'course-agentic-demo' trace.")

Please provide the notebook content (or a link/text excerpt), and I’ll summarize its agent primitives in 5 bullets.

Open https://platform.openai.com/traces to inspect the 'course-agentic-demo' trace.


## Appendix — hosted tools via the plain Responses API (no SDK)

You don't always need the Agents SDK. Hosted tools like `web_search` also work directly on `client.responses.create()`. This is the lighter-weight path for single-shot tool use.

In [9]:
from openai import OpenAI
client = OpenAI()

response = client.responses.create(
    model="gpt-5.6-sol",
    tools=[{"type": "web_search"}],
    input="What are the latest news on AI tool releases? Give me a top-5 list.",
    reasoning={"effort": "medium"},
)
print(response.output_text)

Here’s a **top-5 “latest AI tool releases” list as of June 9, 2026**, ranked by a mix of recency and likely impact:

1. **Apple unveils “Siri AI” at WWDC 2026** — Apple’s long-delayed Siri overhaul was announced on **June 8**, turning Siri into a more conversational, context-aware assistant with personal-data awareness, deeper app integration, a dedicated Siri app, and Dynamic Island integration. It’s expected in beta later this year. ([techcrunch.com](https://techcrunch.com/2026/06/08/apples-long-awaited-ai-siri-overhaul-is-finally-here/))

2. **Microsoft ships a major agent/platform wave at Build 2026** — Microsoft announced **Microsoft Scout**, new Microsoft IQ/Work IQ/Web IQ context layers for enterprise agents, a family of **seven MAI models** including MAI-Thinking-1, MAI-Image-2.5, MAI-Code-1, and a preview GitHub Copilot desktop app for orchestrating coding agents. ([blogs.microsoft.com](https://blogs.microsoft.com/blog/2026/06/02/microsoft-build-2026-be-yourself-at-work/))

3.